# LexAI — Fine-tuning d'embeddings juridiques (Sprint 3)

**Objectif :** Fine-tuner `paraphrase-multilingual-MiniLM-L12-v2` sur 7 257 paires (question juridique, article de loi) pour produire des embeddings spécialisés en droit français.

**Avant de commencer :**
- `Exécution` → `Modifier le type d'exécution` → GPU (**A100** recommandé, T4 fonctionne aussi)
- Uploader `training_pairs_v3.json` quand demandé (cellule 3)


In [1]:
# ── Cellule 1 : Installation ──────────────────────────────────────────────────
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers>=3.0',
    'datasets',
    'accelerate',
])

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {gpu_name} ({gpu_mem:.1f} GB)')
    BATCH_SIZE = 128 if gpu_mem > 35 else 64
else:
    print('ATTENTION : pas de GPU détecté — activer dans Exécution > Modifier le type d\'exécution')
    BATCH_SIZE = 16

print(f'Batch size : {BATCH_SIZE}')

GPU : NVIDIA A100-SXM4-80GB (85.1 GB)
Batch size : 128


In [2]:
# ── Cellule 2 : Google Drive (pour sauvegarder le modèle) ────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/LexAI/lexai-embeddings'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Modèle sauvegardé dans : {SAVE_DIR}')

Mounted at /content/drive
Modèle sauvegardé dans : /content/drive/MyDrive/LexAI/lexai-embeddings


In [3]:
# ── Cellule 3 : Upload du dataset ─────────────────────────────────────────────
from google.colab import files

print('Sélectionnez training_pairs_v3.json depuis votre ordinateur...')
uploaded = files.upload()  # <- sélectionner training_pairs_v3.json

Sélectionnez training_pairs_v3.json depuis votre ordinateur...


Saving training_pairs_v3.json to training_pairs_v3.json


In [4]:
# ── Cellule 4 : Chargement et split train/eval ────────────────────────────────
import json, random

with open('training_pairs_v3.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

random.seed(42)
random.shuffle(raw_data)

split      = int(len(raw_data) * 0.9)
train_data = raw_data[:split]
eval_data  = raw_data[split:]

print(f'Total paires  : {len(raw_data)}')
print(f'Entraînement  : {len(train_data)}')
print(f'Évaluation    : {len(eval_data)}')
print(f'\nExemple paire :')
print(f'  Query    : {train_data[0]["query"]}')
print(f'  Positive : {train_data[0]["positive"][:120]}...')

Total paires  : 7257
Entraînement  : 6531
Évaluation    : 726

Exemple paire :
  Query    : Quelles sont les raisons pour lesquelles une partie peut être déclarée irrecevable dans une procédure civile ?
  Positive : Constitue une fin de non-recevoir tout moyen qui tend à faire déclarer l'adversaire irrecevable en sa demande, sans exam...


In [5]:
# ── Cellule 5 : Préparation des datasets HuggingFace ─────────────────────────
from datasets import Dataset
from sentence_transformers.evaluation import InformationRetrievalEvaluator

train_dataset = Dataset.from_dict({
    'anchor':   [d['query']    for d in train_data],
    'positive': [d['positive'] for d in train_data],
})

queries       = {str(i): d['query']    for i, d in enumerate(eval_data)}
corpus        = {str(i): d['positive'] for i, d in enumerate(eval_data)}
relevant_docs = {str(i): {str(i)}      for i in range(len(eval_data))}

evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name='lexai-juridique',
)

print('Datasets prêts.')


Datasets prêts.


/tmp/ipykernel_3463/3850602702.py:3: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


In [6]:
# ── Cellule 6 : Baseline (modèle généraliste AVANT fine-tuning) ───────────────
from sentence_transformers import SentenceTransformer

BASE_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(BASE_MODEL)

print('=== BASELINE (avant fine-tuning) ===')
baseline_results = evaluator(model)
print(baseline_results)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

=== BASELINE (avant fine-tuning) ===
{'lexai-juridique_cosine_accuracy@1': 0.5013774104683195, 'lexai-juridique_cosine_accuracy@3': 0.6942148760330579, 'lexai-juridique_cosine_accuracy@5': 0.7506887052341598, 'lexai-juridique_cosine_accuracy@10': 0.8195592286501377, 'lexai-juridique_cosine_precision@1': 0.5013774104683195, 'lexai-juridique_cosine_precision@3': 0.23140495867768596, 'lexai-juridique_cosine_precision@5': 0.15013774104683195, 'lexai-juridique_cosine_precision@10': 0.08195592286501376, 'lexai-juridique_cosine_recall@1': 0.5013774104683195, 'lexai-juridique_cosine_recall@3': 0.6942148760330579, 'lexai-juridique_cosine_recall@5': 0.7506887052341598, 'lexai-juridique_cosine_recall@10': 0.8195592286501377, 'lexai-juridique_cosine_ndcg@10': 0.6606173884039905, 'lexai-juridique_cosine_mrr@10': 0.609688880143425, 'lexai-juridique_cosine_map@100': 0.6157609804193432}


In [7]:
# ── Cellule 7 : Fine-tuning ───────────────────────────────────────────────────
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss

loss = MultipleNegativesRankingLoss(model)

NUM_EPOCHS  = 25
steps_total = (len(train_dataset) // BATCH_SIZE) * NUM_EPOCHS
warmup_steps = int(steps_total * 0.1)

print(f'Steps total  : {steps_total}')
print(f'Warmup steps : {warmup_steps}')
print(f'Epochs       : {NUM_EPOCHS}')

args = SentenceTransformerTrainingArguments(
    output_dir='/content/lexai-embeddings-checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    warmup_steps=warmup_steps,
    fp16=True,
    bf16=False,
    learning_rate=2e-5,
    save_strategy='epoch',
    logging_steps=50,
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='lexai-juridique_cosine_ndcg@10',
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=evaluator,
)

trainer.train()
print('\nFine-tuning terminé !')

/tmp/ipykernel_3463/987233291.py:3: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss


Steps total  : 1275
Warmup steps : 127
Epochs       : 25


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss,Lexai-juridique Cosine Accuracy@1,Lexai-juridique Cosine Accuracy@3,Lexai-juridique Cosine Accuracy@5,Lexai-juridique Cosine Accuracy@10,Lexai-juridique Cosine Precision@1,Lexai-juridique Cosine Precision@3,Lexai-juridique Cosine Precision@5,Lexai-juridique Cosine Precision@10,Lexai-juridique Cosine Recall@1,Lexai-juridique Cosine Recall@3,Lexai-juridique Cosine Recall@5,Lexai-juridique Cosine Recall@10,Lexai-juridique Cosine Ndcg@10,Lexai-juridique Cosine Mrr@10,Lexai-juridique Cosine Map@100
1,1.056632,No log,0.629477,0.847107,0.881543,0.933884,0.629477,0.282369,0.176309,0.093388,0.629477,0.847107,0.881543,0.933884,0.790985,0.744391,0.747073
2,0.524256,No log,0.685950,0.873278,0.910468,0.953168,0.685950,0.291093,0.182094,0.095317,0.685950,0.873278,0.910468,0.953168,0.829863,0.789317,0.791329
3,0.363382,No log,0.702479,0.881543,0.926997,0.966942,0.702479,0.293848,0.185399,0.096694,0.702479,0.881543,0.926997,0.966942,0.843155,0.802686,0.804090
4,0.263229,No log,0.695592,0.892562,0.926997,0.966942,0.695592,0.297521,0.185399,0.096694,0.695592,0.892562,0.926997,0.966942,0.842359,0.801264,0.802726
5,0.209630,No log,0.717631,0.899449,0.936639,0.971074,0.717631,0.299816,0.187328,0.097107,0.717631,0.899449,0.936639,0.971074,0.854132,0.815592,0.816811
6,0.176120,No log,0.714876,0.896694,0.940771,0.971074,0.714876,0.298898,0.188154,0.097107,0.714876,0.896694,0.940771,0.971074,0.854197,0.815502,0.816972
7,0.147694,No log,0.725895,0.903581,0.943526,0.975207,0.725895,0.301194,0.188705,0.097521,0.725895,0.903581,0.943526,0.975207,0.859994,0.821955,0.822958
8,0.143350,No log,0.724518,0.911846,0.944904,0.971074,0.724518,0.303949,0.188981,0.097107,0.724518,0.911846,0.944904,0.971074,0.859583,0.822462,0.823804
9,0.114628,No log,0.732782,0.909091,0.946281,0.972452,0.732782,0.303030,0.189256,0.097245,0.732782,0.909091,0.946281,0.972452,0.863097,0.826802,0.828098
10,0.108768,No log,0.730028,0.911846,0.949036,0.975207,0.730028,0.303949,0.189807,0.097521,0.730028,0.911846,0.949036,0.975207,0.863248,0.826183,0.827256


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Fine-tuning terminé !


In [8]:
# ── Cellule 8 : Résultats APRÈS fine-tuning ───────────────────────────────────
print('=== APRÈS FINE-TUNING ===')
ft_results = evaluator(model)
print(ft_results)

print('\n=== COMPARAISON ===')
for k in ft_results:
    if k in baseline_results:
        delta = ft_results[k] - baseline_results[k]
        print(f'{k:<50} : {baseline_results[k]:.4f} → {ft_results[k]:.4f}  (Δ {delta:+.4f})')

=== APRÈS FINE-TUNING ===
{'lexai-juridique_cosine_accuracy@1': 0.7300275482093664, 'lexai-juridique_cosine_accuracy@3': 0.9035812672176309, 'lexai-juridique_cosine_accuracy@5': 0.9504132231404959, 'lexai-juridique_cosine_accuracy@10': 0.977961432506887, 'lexai-juridique_cosine_precision@1': 0.7300275482093664, 'lexai-juridique_cosine_precision@3': 0.3011937557392103, 'lexai-juridique_cosine_precision@5': 0.19008264462809912, 'lexai-juridique_cosine_precision@10': 0.09779614325068868, 'lexai-juridique_cosine_recall@1': 0.7300275482093664, 'lexai-juridique_cosine_recall@3': 0.9035812672176309, 'lexai-juridique_cosine_recall@5': 0.9504132231404959, 'lexai-juridique_cosine_recall@10': 0.977961432506887, 'lexai-juridique_cosine_ndcg@10': 0.8632784816319049, 'lexai-juridique_cosine_mrr@10': 0.8254788141151779, 'lexai-juridique_cosine_map@100': 0.826348634238287}

=== COMPARAISON ===
lexai-juridique_cosine_accuracy@1                  : 0.5014 → 0.7300  (Δ +0.2287)
lexai-juridique_cosine_accu

In [9]:
# ── Cellule 9 : Sauvegarde dans Google Drive ──────────────────────────────────
model.save(SAVE_DIR)
print(f'Modèle sauvegardé : {SAVE_DIR}')

# Sauvegarde des métriques pour le rapport
import json
metrics = {
    'baseline': baseline_results,
    'finetuned': ft_results,
    'model_base': BASE_MODEL,
    'train_pairs': len(train_data),
    'epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
}
with open(f'{SAVE_DIR}/finetune_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Métriques sauvegardées dans finetune_metrics.json')
print('\nPour utiliser le modèle dans LexAI :')
print(f'  model = SentenceTransformer("{SAVE_DIR}")')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle sauvegardé : /content/drive/MyDrive/LexAI/lexai-embeddings
Métriques sauvegardées dans finetune_metrics.json

Pour utiliser le modèle dans LexAI :
  model = SentenceTransformer("/content/drive/MyDrive/LexAI/lexai-embeddings")


In [10]:
# ── Cellule 10 : Test qualitatif rapide ───────────────────────────────────────
import torch
from sentence_transformers import util

test_queries = [
    'Peut-on licencier une femme enceinte ?',
    'Quelles sont les conditions du mariage en France ?',
    'Qu\'est-ce que la légitime défense ?',
]

# Encoder quelques articles de référence (premiers du dataset eval)
sample_docs  = [d['positive'] for d in eval_data[:50]]
doc_embeddings = model.encode(sample_docs, convert_to_tensor=True)

print('=== TEST QUALITATIF ===')
for q in test_queries:
    q_emb = model.encode(q, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, doc_embeddings)[0]
    top = scores.topk(1)
    best_doc = sample_docs[top.indices[0]]
    print(f'\nQ: {q}')
    print(f'   Score: {top.values[0]:.4f}')
    print(f'   Doc:   {best_doc[:100]}...')

=== TEST QUALITATIF ===

Q: Peut-on licencier une femme enceinte ?
   Score: 0.4221
   Doc:   Il est interdit d'employer la salariée pendant une période de huit semaines au total avant et après ...

Q: Quelles sont les conditions du mariage en France ?
   Score: 0.3156
   Doc:   L'acte de mariage énoncera : 1° Les prénoms, noms, professions, âges, dates et lieux de naissance, d...

Q: Qu'est-ce que la légitime défense ?
   Score: 0.1668
   Doc:   Les exceptions doivent, à peine d'irrecevabilité, être soulevées simultanément et avant toute défens...
